# Hour 3 · From a PDF to a research database

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/3d_udb_pdf_to_sqlite.ipynb)

**Goal:** turn a PDF that you obtained yourself into a local SQLite database, inspect the extraction, and ask reproducible questions with SQL.

> The workshop does not provide or download the UDB PDF and does not collect your generated database. Keep both files local and do not share them without separate authorization from the rights holders.

## 0. Setup

This notebook uses the workshop's checked-in parser. In Colab you will be asked to upload your own PDF into the temporary runtime.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = Path("/content/ugarit-dh-workshop")
    if not REPO_DIR.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pdfminer.six"], check=True)
    os.chdir(REPO_DIR / "notebooks")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
LOCAL_DATA = ROOT / "local_data"
LOCAL_DATA.mkdir(exist_ok=True)
PDF_PATH = LOCAL_DATA / "Ugaritic_data_bank.pdf"
DB_PATH = LOCAL_DATA / "udb.sqlite"
print("Local workspace:", LOCAL_DATA)


## 1. Supply the PDF locally

In Colab, running the next cell opens an upload dialog. Locally, place the file at `local_data/Ugaritic_data_bank.pdf` and run the cell.

In [ ]:
if IN_COLAB and not PDF_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one PDF")
    uploaded_name = next(iter(uploaded))
    Path(uploaded_name).replace(PDF_PATH)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Place your PDF at {PDF_PATH}")
print(f"Ready: {PDF_PATH.name} ({PDF_PATH.stat().st_size / 1024 / 1024:.1f} MiB)")


## 2. Parse and build SQLite

This is the computationally expensive step. The parser repairs the PDF's custom character encoding, separates tablet metadata from readings, and extracts sources, literature, and tablet-level comments.

In [ ]:
from workshop_tools.build_udb_sqlite import build_database

counts = build_database(PDF_PATH, DB_PATH, overwrite=True)
counts


## 3. Inspect what was extracted

A good digital-humanities workflow checks the transformation before analyzing it.

In [ ]:
import sqlite3
import pandas as pd

connection = sqlite3.connect(DB_PATH)
pd.read_sql_query("""
SELECT tablet, ref, reader, text, comment
FROM readings
WHERE tablet = '1.16'
ORDER BY reading_id
LIMIT 12
""", connection)


In [ ]:
pd.read_sql_query("""
SELECT scope_type, scope, title, container_title, citation
FROM literature
WHERE tablet = '1.16'
ORDER BY literature_id
LIMIT 12
""", connection)


## 4. Ask a research question

The query below finds places where editors give different readings for the same reference.

In [ ]:
pd.read_sql_query("""
SELECT tablet, ref,
       COUNT(DISTINCT reader) AS readers,
       COUNT(DISTINCT text) AS distinct_readings
FROM readings
GROUP BY tablet, ref
HAVING COUNT(DISTINCT text) > 1
ORDER BY distinct_readings DESC, tablet, ref
LIMIT 20
""", connection)


## Your turn

Change one thing and rerun:

1. Replace `1.16` with another tablet.
2. Search `readings.text` for a form such as `bʿl` or `mlk`.
3. Count bibliography categories from `literature.categories_json`.
4. Compare a database result with the PDF. What information did the parser preserve, and what still needs human checking?

When finished, close the connection. In Colab the uploaded PDF and generated database disappear with the runtime; locally they remain under ignored `local_data/`.

In [ ]:
connection.close()
print("Finished. Do not upload or commit local_data/.")
